In [1]:
!pip install "tensorboard~=2.19.0"


In [2]:
import tensorflow as tf, tensorboard as tb
print(tf.__version__)
print(tb.__version__)


2.19.0
2.19.0


In [3]:
!pip install --upgrade pip
!pip install --upgrade datasets[audio] transformers accelerate evaluate jiwer gradio datasets pyarrow


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 191.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 141.3 MB/s  0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.21.4
    Uninstalling tokenizers-0.21.4:
      Successfully uninstalled tokenizers-0.21.4
  Attempting uninstall: transformers
    Found existing installation: transformers 4.52.0
    Uninstalling transformers-4.52.0:
      Successfully uninstalled transformers-4.52.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [transformers]


In [4]:
!pip install -U torchaudio torchvision


In [5]:
!pip install -U torchcodec torch

In [6]:
!pip install transformers==4.52.0 --quiet

Reason for being yanked: <none given>


In [7]:
import torch, torchaudio, torchvision
print(torch.__version__)
print(torchaudio.__version__)
print(torchvision.__version__)


2.9.1+cu128
2.9.1+cu128
0.24.1+cu128


In [8]:

import datasets
print(datasets.__version__)



4.4.2


In [9]:
import transformers
print(transformers.__version__)


4.52.0


In [70]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
datasetdict = load_dataset("ysdede/commonvoice_17_tr_fixed")

In [71]:
print(datasetdict)

DatasetDict({
    train: Dataset({
        features: ['audio', 'transcription', 'duration', 'up_votes', 'down_votes', 'age', 'gender', 'accent'],
        num_rows: 26501
    })
    test: Dataset({
        features: ['audio', 'transcription', 'duration', 'up_votes', 'down_votes', 'age', 'gender', 'accent'],
        num_rows: 9650
    })
    validation: Dataset({
        features: ['audio', 'transcription', 'duration', 'up_votes', 'down_votes', 'age', 'gender', 'accent'],
        num_rows: 8639
    })
    validated: Dataset({
        features: ['audio', 'transcription', 'duration', 'up_votes', 'down_votes', 'age', 'gender', 'accent'],
        num_rows: 46345
    })
})


In [72]:
datasetdict = datasetdict.rename_column("transcription", "sentence")

In [73]:
datasetdict = datasetdict.select_columns(["audio", "sentence"])

In [74]:
datasetdict

DatasetDict({
    train: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 26501
    })
    test: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 9650
    })
    validation: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 8639
    })
    validated: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 46345
    })
})

In [75]:
print(datasetdict["train"][0])

{'audio': <datasets.features._torchcodec.AudioDecoder object at 0x7b972c4df4d0>, 'sentence': 'Kosova başkentinswki yolcu sayısı arttı.'}


In [76]:
from datasets import Audio
datasetdict = datasetdict.cast_column("audio", Audio(sampling_rate=16000))

In [77]:
from transformers import WhisperFeatureExtractor

feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-medium")

In [78]:
from transformers import WhisperTokenizer

tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-medium", language="turkish", task="transcribe")


In [20]:
datasetdict

DatasetDict({
    train: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 26501
    })
    test: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 9650
    })
    validation: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 8639
    })
    validated: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 46345
    })
})

In [79]:
del datasetdict["validation"]
del datasetdict["validated"]

In [80]:
input_str = datasetdict["train"][0]["sentence"]
labels = tokenizer(input_str).input_ids
decoded_with_special = tokenizer.decode(labels, skip_special_tokens=False)
decoded_str = tokenizer.decode(labels, skip_special_tokens=True)

print(f"Input:                 {input_str}")
print(f"Decoded w/ special:    {decoded_with_special}")
print(f"Decoded w/out special: {decoded_str}")
print(f"Are equal:             {input_str == decoded_str}")

Input:                 Kosova başkentinswki yolcu sayısı arttı.
Decoded w/ special:    <|startoftranscript|><|tr|><|transcribe|><|notimestamps|>Kosova başkentinswki yolcu sayısı arttı.<|endoftext|>
Decoded w/out special: Kosova başkentinswki yolcu sayısı arttı.
Are equal:             True


In [81]:
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained("openai/whisper-medium", language="turkish", task="transcribe")


In [82]:
print(datasetdict["train"][0])

{'audio': <datasets.features._torchcodec.AudioDecoder object at 0x7b97a70af440>, 'sentence': 'Kosova başkentinswki yolcu sayısı arttı.'}


In [98]:
from transformers import WhisperProcessor

BASE_MODEL_ID = "openai/whisper-medium"   # base karşılaştırma için
processor = WhisperProcessor.from_pretrained(BASE_MODEL_ID)

feature_extractor = processor.feature_extractor
tokenizer = processor.tokenizer

In [83]:
def prepare_dataset(batch):
    # load and resample audio data from 48 to 16kHz
    audio = batch["audio"]

    # compute log-Mel input features from input audio array
    batch["input_features"] = feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]

    # encode target text to label ids
    batch["labels"] = tokenizer(batch["sentence"]).input_ids
    return batch


In [85]:
datasetdict = datasetdict.map(prepare_dataset, remove_columns=datasetdict.column_names["train"], num_proc=4)


Map (num_proc=4):   0%|          | 0/26501 [00:00<?, ? examples/s]

KeyError: 'audio'

In [95]:
import torch

def data_collator_speech_seq2seq(features):
    # input_features -> torch tensor
    input_features = torch.tensor([f["input_features"] for f in features], dtype=torch.float32)

    # labels padding
    label_features = [{"input_ids": f["labels"]} for f in features]
    labels_batch = tokenizer.pad(label_features, return_tensors="pt")

    labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

    return {
        "input_features": input_features,
        "labels": labels,
    }


In [96]:
import evaluate
wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # -100 olan yerleri pad token ile değiştir (decode edebilmek için)
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}


In [100]:
from transformers import WhisperForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer

base_model = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL_ID)

# İstersen TR transkripsiyon için:
# base_model.generation_config.language = "turkish"
# base_model.generation_config.task = "transcribe"

args = Seq2SeqTrainingArguments(
    output_dir="./whisper_base_eval",
    per_device_eval_batch_size=8,
    predict_with_generate=True,   # WER için generate şart
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=base_model,
    args=args,
    eval_dataset=datasetdict["test"],   # ya da validation splitin
    data_collator=data_collator_speech_seq2seq,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

base_metrics = trainer.evaluate()
print(base_metrics)


/tmp/ipython-input-329899822.py:17: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
Due to a bug fix in https://github.com/huggingface/transformers/pull/28687 transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English.This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.43.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from

{'eval_loss': 3.1742212772369385, 'eval_model_preparation_time': 0.0105, 'eval_wer': 0.2946083345267077, 'eval_runtime': 3823.2254, 'eval_samples_per_second': 2.524, 'eval_steps_per_second': 0.316}


In [102]:
import torch
from transformers import WhisperForConditionalGeneration

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

FINETUNED_PATH = "/content/drive/MyDrive/bitirme_projesi/whisper_model"
ft_model = WhisperForConditionalGeneration.from_pretrained(FINETUNED_PATH).to(device)
ft_model.eval()

trainer.model = ft_model
ft_metrics = trainer.evaluate()
print(ft_metrics)


A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.


{'eval_loss': 2.078791379928589, 'eval_model_preparation_time': 0.0105, 'eval_wer': 0.17291994844622655, 'eval_runtime': 3782.85, 'eval_samples_per_second': 2.551, 'eval_steps_per_second': 0.319}
